### REVIEW AGENT

Retrieved Products ---> Retrieved customer reviews ---> Summarize review using LLM ---> pros , cons, sentiment

In [1]:
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import json

import os

load_dotenv()

True

In [2]:
#loading sample datasets: 

reviews_df  = pd.read_parquet("../Data/Cleaned/reviews_sample.parquet")


In [3]:
reviews_df.head(20)

,rating,product_title,review_title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review_date,price,average_rating,rating_number
0,4.0,"Honsky Thumbs-up Phone Stand for Tablets, E-re...",Four Stars,Stocking stuffers for my boys & they liked them.,[],B00UKXP6Y2,B07CKPPSB4,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,1485870336000,0,True,2017-01-31 13:45:36.000,19.99,4.4,4307
1,5.0,Bling Diamond Case for Samsung Galaxy S10 Plus...,"Love, love and I get so many compliments!",Love this.....it's amazing and haven't lost an...,[],B07P9544RP,B09JSXJ7X3,AGBUJDDLRJIUJFTKPABJT6CJTHRQ,1576168322121,1,True,2019-12-12 16:32:02.121,11.98,4.3,1478
2,4.0,"SHENMZ LG V20 Battery,6800mAh (More Than 2X Ex...",Seein is believing that's only proof by one yo...,Really nothing I don't have to charge in a cou...,[],B07MW7ZQVN,B07MW7ZQVN,AFUZ5E7QAHSVTLIY253GFAYZPN2Q,1611422480932,0,True,2021-01-23 17:21:20.932,25.95,4.3,1109
3,5.0,Unov Pixel 3 Case Clear Soft TPU Shock Absorpt...,More durable than expected,Beautiful paint and stays on very well. Seems ...,[],B07XF34VRB,B07XH1QWJ7,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,1611191271232,0,True,2021-01-21 01:07:51.232,8.39,4.5,1807
4,5.0,"USB Plug, USB Wall Charger 3 Pack, GiGreen Dua...",Does the job,Works fine.,[],B077B8D1S6,B0B5XKXVDZ,AE3Q6AEWP7Y7CH4N6IWEP4YBNP2A,1646078603429,0,True,2022-02-28 20:03:23.429,10.99,4.7,4536
5,1.0,Supershieldz Designed for LG Stylo 2 Tempered ...,DONT WASTE YOUR TIME BUY THE CHEAPER ONES,i was hoping for better results ... its a litt...,[],B01EBI0LQY,B01EBI0LQY,AHGAOIZVODNHYMNCBV4DECZH42UQ,1468268873000,2,True,2016-07-11 20:27:53.000,7.49,4.3,1279
6,5.0,"FYY Case for iPhone 11 6.1”, Luxury [Cowhide G...",Quality item,Beautiful high quality phone wallet. The magn...,[],B07X7YL4TL,B094FSS89F,AHZ6XMOLEWA67S3TX7IWEXXGWSOA,1582758366816,0,True,2020-02-26 23:06:06.816,19.99,4.6,1540
7,5.0,"Yacig Capacitive Stylus Pen, 4-in-1 High Sensi...",4 in 1 stylus,I purchased this stylus right before I purchas...,[],B07GKY9FJZ,B087FFK6C3,AFZUK3MTBIBEDQOPAK3OATUOUKLA,1673289121448,1,True,2023-01-09 18:32:01.448,12.99,4.4,4044
8,5.0,"OWNITOW Nylon Watch Bands, Canvas Fabric Balli...",bright orange watch band,My husband is all about safety colors and this...,[],B07KW8TVD7,B08GK8FP2C,AFZUK3MTBIBEDQOPAK3OATUOUKLA,1565065572422,0,True,2019-08-06 04:26:12.422,13.99,4.4,1577
9,5.0,"hii Cable Card, Portable Phone Charger, All-in...",Great versatility,So much versatility,[],B09193DK97,B09193DK97,AGF42GID7QWDCNFTJRCTMKAITJJA,1641674049229,0,True,2022-01-08 20:34:09.229,NaN,3.9,24


In [4]:
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature=0
)

In [5]:
reviews_df[["parent_asin"]].head()

,parent_asin
0,B07CKPPSB4
1,B09JSXJ7X3
2,B07MW7ZQVN
3,B07XH1QWJ7
4,B0B5XKXVDZ


In [6]:
parent_asin = "B09JSXJ7X3"

In [7]:
product_reviews = reviews_df[
    reviews_df["parent_asin"] == parent_asin
]

In [8]:
print(product_reviews.shape)
product_reviews.head()

(259, 15)


,rating,product_title,review_title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review_date,price,average_rating,rating_number
1,5.0,Bling Diamond Case for Samsung Galaxy S10 Plus...,"Love, love and I get so many compliments!",Love this.....it's amazing and haven't lost an...,[],B07P9544RP,B09JSXJ7X3,AGBUJDDLRJIUJFTKPABJT6CJTHRQ,1576168322121,1,True,2019-12-12 16:32:02.121,11.98,4.3,1478
793,5.0,Bling Diamond Case for Samsung Galaxy S10 Plus...,Helpful Review,I used my case for a SAMSUNG GALAXY 10+. Supe...,[],B07PBYN297,B09JSXJ7X3,AHURHCFALILO5QFBMPSY3VJIUW4A,1580509735679,0,True,2020-01-31 22:28:55.679,11.98,4.3,1478
1348,1.0,Bling Diamond Case for Samsung Galaxy S10 Plus...,False advertisement!,The rhinestones on the ACTUAL phone case that ...,[],B07P6W5RGG,B09JSXJ7X3,AHEICJLV723C5MIECGYYDTDYFJUQ,1620748763455,4,True,2021-05-11 15:59:23.455,11.98,4.3,1478
2404,5.0,Bling Diamond Case for Samsung Galaxy S10 Plus...,So sparkly!,I LOVE THIS SO MUCH! my only con would be if ...,[],B07P9544RP,B09JSXJ7X3,AGKTZ3QCBZRIPFHJ62AMZENNRQWQ,1603990837893,0,True,2020-10-29 17:00:37.893,11.98,4.3,1478
2712,5.0,Bling Diamond Case for Samsung Galaxy S10 Plus...,Really Gorgeous case!!!,Really gorgeous i did drop it once & got a chi...,[],B07PBYN297,B09JSXJ7X3,AGHCCRLZS4GNIN44OQ4EA7G2D4HA,1589825108244,0,True,2020-05-18 18:05:08.244,11.98,4.3,1478


Checking the retrieved reviews:


In [9]:
print("Number of Reviews: ", len(product_reviews))

product_reviews[["parent_asin", "rating", "text"]].head(10)

Number of Reviews:  259


,parent_asin,rating,text
1,B09JSXJ7X3,5.0,Love this.....it's amazing and haven't lost an...
793,B09JSXJ7X3,5.0,I used my case for a SAMSUNG GALAXY 10+. Supe...
1348,B09JSXJ7X3,1.0,The rhinestones on the ACTUAL phone case that ...
2404,B09JSXJ7X3,5.0,I LOVE THIS SO MUCH! my only con would be if ...
2712,B09JSXJ7X3,5.0,Really gorgeous i did drop it once & got a chi...
2770,B09JSXJ7X3,1.0,Let me start off with the case itself was cute...
5087,B09JSXJ7X3,5.0,Myself personally liked everything about this ...
6978,B09JSXJ7X3,2.0,Not a shiny bling but cute. Very heavy actuall...
9125,B09JSXJ7X3,5.0,"Awsome product! Made very well, it is shiny gl..."
11313,B09JSXJ7X3,4.0,Looks nice but very poor quality


In [10]:
reviews_text = "\n\n".join(

    product_reviews["text"]
    .dropna()
    .head(20)
    .tolist()

)

In [11]:
print(reviews_text[:1000])

Love this.....it's amazing and haven't lost any gems.  When others see it they want it!  So I tell them where to buy it!!!!

I used my case for a SAMSUNG GALAXY 10+.  Super cute!  Definitely do not regret this purchase.  Please note, there is no ring, it's a flat case.

The rhinestones on the ACTUAL phone case that you're paying for are all the same size, not different sized rhinestones like their ad photos display! That's the whole reason I bought this case, FOR THE WAY IT LOOKED!!!<br />Just take photos of the actual product that you're selling so that customers know what they are getting!

I LOVE THIS SO MUCH!  my only con would be if they could put a pop socket or ring in it or leave room for one.

Really gorgeous i did drop it once & got a chip on the case but it did protect my phone!

Let me start off with the case itself was cute. The jewels started to fall off later that day. I wanted to live this phone case but it fell apart within one month 😔. Fits well on phone

Myself perso

Bulding prompt for LLM 

In [12]:
prompt = f"""
You are an ecommerce product analyst.

Read the customer reviews below.

Summarize them into the following sections:

1. Pros
2. Cons
3.Overall customer sentiment

Reviews:

{reviews_text}
"""

In [13]:
response = llm.invoke(prompt)

In [14]:
print(response.content)

### 1. Pros
- Many customers find the case visually appealing and cute, often receiving compliments from others.
- The case fits well on various phone models, including the Samsung Galaxy 10+.
- Some users appreciate the sturdiness and protective qualities of the case, noting it can withstand drops.
- A few reviews highlight the quality of the materials, with mentions of shiny glass crystals and well-glued stones.
- Customers express a desire to purchase additional cases, indicating satisfaction with the product.

### 2. Cons
- Several reviews mention that the rhinestones fall off easily, with some users experiencing this issue within a short period of use.
- The actual product does not match the advertised images, particularly regarding the size and variety of rhinestones.
- Some customers find the case to be heavy and bulky, which may not be to everyone's liking.
- A few users reported that the case chipped or showed signs of wear after minimal use.

### 3. Overall Customer Sentiment

Creating review agent

In [15]:
from pydantic import BaseModel

class ReviewSummary(BaseModel):
    pros: list[str]
    cons: list[str]
    overall_sentiment: str
    recommended_for: str
    avoid_if: str
    summary: str


class ReviewAgent:

    def __init__(self, llm, reviews_df):

        self.llm = llm
        self.reviews_df = reviews_df
        self.cache = {}

        # Structured LLM
        self.structured_llm = llm.with_structured_output(
            ReviewSummary
        )

    def summarize_reviews(
        self,
        parent_asin: str,
        max_reviews: int = 20
    ) -> dict:

        # -----------------------------
        # Cache
        # -----------------------------

        if parent_asin in self.cache:
            return self.cache[parent_asin]

        # -----------------------------
        # Retrieve Reviews
        # -----------------------------

        product_reviews = self.reviews_df[
            self.reviews_df["parent_asin"] == parent_asin
        ]

        # -----------------------------
        # No reviews available
        # -----------------------------

        if product_reviews.empty:

            result = {
                "parent_asin": parent_asin,
                "title": "Unknown Product",
                "review_count": 0,
                "average_rating": 0,
                "pros": [],
                "cons": [],
                "overall_sentiment": "Unknown",
                "recommended_for": "",
                "avoid_if": "",
                "summary": "No customer reviews available.",
                "sample_reviews": ""
            }

            self.cache[parent_asin] = result

            return result

        # -----------------------------
        # Sort Reviews
        # -----------------------------

        product_reviews = product_reviews.sort_values(
            by="rating",
            ascending=False
        )

        # -----------------------------
        # Statistics
        # -----------------------------

        average_rating = round(
            product_reviews["rating"].mean(),
            2
        )

        review_count = len(product_reviews)

        title = product_reviews.iloc[0].get(
            "product_title",
            "Unknown Product"
        )

        # -----------------------------
        # Combine Reviews
        # -----------------------------

        reviews_text = "\n\n".join(

            product_reviews["text"]
            .dropna()
            .head(max_reviews)
            .tolist()

        )

        # -----------------------------
        # Prompt
        # -----------------------------

        prompt = f"""
You are an expert ecommerce product analyst.

Product Title:
{title}

Average Rating:
{average_rating}

Number of Reviews:
{review_count}

Read the customer reviews below.

Base your answer ONLY on the reviews provided.

Identify:

• Pros
• Cons
• Overall Sentiment
• Recommended For
• Avoid If
• Short Summary

Customer Reviews:

{reviews_text}
"""

        # -----------------------------
        # LLM
        # -----------------------------

        try:

            summary = self.structured_llm.invoke(prompt)

        except Exception:

            summary = ReviewSummary(
                pros=[],
                cons=[],
                overall_sentiment="Unknown",
                recommended_for="",
                avoid_if="",
                summary="Review summarization failed."
            )

        # -----------------------------
        # Final Output
        # -----------------------------

        result = {
            "parent_asin": parent_asin,
            "title": title,
            "review_count": review_count,
            "average_rating": average_rating,
            "pros": summary.pros,
            "cons": summary.cons,
            "overall_sentiment": summary.overall_sentiment,
            "recommended_for": summary.recommended_for,
            "avoid_if": summary.avoid_if,
            "summary": summary.summary,
            "sample_reviews": reviews_text
        }

        # -----------------------------
        # Cache
        # -----------------------------

        self.cache[parent_asin] = result

        return result

In [16]:
review_agent = ReviewAgent(
    llm,
    reviews_df
)

result = review_agent.summarize_reviews(parent_asin,max_reviews=5)

print(result)

{'parent_asin': 'B09JSXJ7X3', 'title': 'Bling Diamond Case for Samsung Galaxy S10 Plus,Aearl 3D Homemade Luxury Sparkle Crystal Rhinestone Shiny Glitter Full Clear Stones Back Phone Cover with Screen Protector for Galaxy S10 Plus-All White', 'review_count': 259, 'average_rating': np.float64(4.24), 'pros': ['Looks amazing and matches the picture', 'Wife loved the gift', "Haven't lost any gems", 'Attracts attention from others', 'Super cute design', 'Holds up well over time'], 'cons': ["No ring included, it's a flat case"], 'overall_sentiment': 'Positive', 'recommended_for': 'Anyone looking for a stylish and eye-catching phone case for the Samsung Galaxy S10 Plus', 'avoid_if': 'You prefer a case with a ring or additional features', 'summary': 'The Bling Diamond Case for Samsung Galaxy S10 Plus is a stylish and attractive phone cover that has received positive feedback for its design and durability. Customers appreciate its appearance and the fact that it holds up well, although it lacks 

In [17]:
sample_asins = (
    reviews_df["parent_asin"]
    .drop_duplicates()
    .sample(10, random_state=42)
)

for asin in sample_asins:

    print("=" * 100)

    result = review_agent.summarize_reviews(asin)

    print("Parent ASIN :", result["parent_asin"])
    print("Title :", result["title"])
    print("Average Rating :", result["average_rating"])
    print("Review Count :", result["review_count"])

    print()

    print("Pros:", result["pros"])

    print()

    print("Cons:", result["cons"])

    print()

    print("Overall Sentiment:", result["overall_sentiment"])

    print()

    print(result["summary"])

    print()

Parent ASIN : B08HYXTWTJ
Title : Inber iPhone 11 Case with Glass Screen Protector,Clear TPU Cover with Fashionable Pink Champagne Flower Floral Designs for Girls Women,Shockproof Protective Phone Case for Apple iPhone 11 6.1"
Average Rating : 5.0
Review Count : 1

Pros: ['Quick delivery', 'Well packaged', 'Accurate appearance as shown in photos', 'Perfect fit for iPhone 11', 'Durable and sturdy design', 'Confidence in phone protection']

Cons: []

Overall Sentiment: Positive

The Inber iPhone 11 Case is a stylish and durable option for girls and women, featuring a fashionable floral design and providing excellent protection for the device.

Parent ASIN : B007635BWG
Title : NOKIA LUMIA 900 SOLID BLACK HYBRID RUBBERISED BACK COVER CASE, IN QUBITS RETAIL PACKAGING
Average Rating : 4.0
Review Count : 1

Pros: ['Good fit', "Doesn't add too much bulk", 'Helps with grip', 'Good value']

Cons: ['Attracts cat hair and fine lint']

Overall Sentiment: Positive

The NOKIA LUMIA 900 Hybrid Rubberis

### Review Agent Completed

This notebook:
- Retrieves reviews using parent_asin
- Summarizes customer reviews using GPT-4o-mini
- Returns:
  - parent_asin
  - title
  - review_count
  - review summary

The Review Agent will later receive parent_asin values from the Retrieval Agent during orchestration.